# InstantMesh — real end-to-end validation run (free Colab T4)

**What this is:** the official `TencentARC/InstantMesh` pipeline (Zero123++ → LRM triplane → FlexiCubes),
run once on a **free** Google Colab T4 GPU, producing a **real** 3D mesh from a **real** photograph.

**Cost:** $0. No paid API, no credits, no card.

**What you do (3 clicks):**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. `Runtime` → **Run all**
3. In the last cell, let it download `instantmesh_real_glb.zip` and **attach that file to the chat**

Every setting is pre-chosen. If a stage fails, the notebook prints which fallback it took and why.

**Input photo:** BMW M5 (F90), fetched automatically from Wikimedia Commons
(*BMW M5 F90 Competition (51585150351).jpg*, © Alexandre Prevot, **CC BY-SA 2.0**). Fetched rather than
bundled so the licence stays with its author.

**Recorded for the audit:** GPU name, peak VRAM, wall-clock time per stage, vertex/triangle counts,
texture size, which export path succeeded, and SHA-256 of both the input photo and the output GLB.

## 1 · Confirm we have a GPU (fails fast)

In [ ]:
import json, platform, sys
info = {"python": platform.python_version(), "platform": platform.platform()}
try:
    import torch
    info.update(torch=torch.__version__, cuda=torch.version.cuda,
                cuda_available=bool(torch.cuda.is_available()))
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info["gpu"] = p.name
        info["vram_total_gb"] = round(p.total_memory / 1024**3, 2)
except Exception as e:
    info["torch_error"] = str(e)

print(json.dumps(info, indent=2))
if not info.get("cuda_available"):
    raise SystemExit("NO GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Run all. "
                     "InstantMesh has no CPU path (official code calls torch.device('cuda')).")
print("\nGPU OK ->", info["gpu"], f"({info['vram_total_gb']} GiB VRAM)")

## 2 · Install dependencies (official pins)

CUDA torch **before** `requirements.txt`, or pip resolves a CPU-only wheel.
`nvdiffrast` compiles from source: 5–10 minutes of silence is expected, not a hang.

In [ ]:
%%bash
set -e
cd /content
[ -d InstantMesh ] || git clone --depth 1 https://github.com/TencentARC/InstantMesh.git
cd InstantMesh
export CUDA_HOME=/usr/local/cuda
python -m pip install -q --upgrade pip
python -m pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu121 \
  || python -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
python -m pip install -q xformers==0.0.22.post7 || echo "xformers pin failed (non-fatal)"
python -m pip install -q -r requirements.txt
python -c "import torch, trimesh, diffusers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| trimesh', trimesh.__version__, '| diffusers', diffusers.__version__)"

## 3 · Pre-fetch weights (~7.3 GB)

Verified sizes (Hub API, 2026-11-01, repo sha `b785b4ec`):
`instant_mesh_large.ckpt` 1,514,818,077 B · `instant_mesh_base.ckpt` 1,253,574,354 B ·
`diffusion_pytorch_model.bin` 1,732,056,502 B. Plus `sudo-ai/zero123plus-v1.2` (~5.4 GB) and
`facebook/dino-vitb16` (~0.33 GB).

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("/content/InstantMesh/ckpts", exist_ok=True)
for f in ["instant_mesh_large.ckpt", "instant_mesh_base.ckpt", "diffusion_pytorch_model.bin"]:
    src = hf_hub_download("TencentARC/InstantMesh", f, repo_type="model")
    dst = f"/content/InstantMesh/ckpts/{f}"
    if not os.path.exists(dst):
        try: os.link(src, dst)
        except OSError: shutil.copyfile(src, dst)
    print(f, os.path.getsize(dst), "bytes")
snapshot_download("sudo-ai/zero123plus-v1.2")
snapshot_download("facebook/dino-vitb16")
print("weights ready")

## 4 · Input photograph (auto-fetched — nothing for you to upload)

Three CC-licensed BMW M5 photographs are tried in order; the first that downloads and decodes wins.
A copy is saved as `udata/input.jpg` and its SHA-256 printed, so the audit can prove
input ↔ output correspondence.

**Use your own photo instead?** Replace this cell with:
```python
from google.colab import files
up = files.upload(); INPUT_IMAGE = "/content/InstantMesh/" + next(iter(up))
```
A clean side-on shot of one object works best. Thin parts (mirrors, wipers, grilles) will be approximate.

In [ ]:
import io, os, hashlib
from PIL import Image
from IPython.display import display
import requests

CANDIDATES = [
    ("https://commons.wikimedia.org/wiki/Special:FilePath/BMW%20M5%20F90%20Competition%20(51585150351).jpg?width=1280",
     "BMW M5 F90 Competition (51585150351).jpg", "Alexandre Prevot", "CC BY-SA 2.0"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/BMW%20F90%20M5%20Competition%20(47090285514).jpg?width=1280",
     "BMW F90 M5 Competition (47090285514).jpg", "Alexandre Prevot", "CC BY-SA 2.0"),
    ("https://commons.wikimedia.org/wiki/Special:FilePath/BMW%20M5%20CS%20(F90%2C%202022)%20-%2052454231629.jpg?width=1280",
     "BMW M5 CS (F90, 2022) - 52454231629.jpg", "Charles", "CC BY-SA 2.0"),
]

os.chdir("/content/InstantMesh")
os.makedirs("udata", exist_ok=True)
INPUT_IMAGE, ATTRIBUTION = None, None
for url, fname, author, lic in CANDIDATES:
    try:
        r = requests.get(url, timeout=60, headers={"User-Agent": "instantmesh-audit/1.0 (research)"})
        r.raise_for_status()
        probe = Image.open(io.BytesIO(r.content)); probe.load()
        if min(probe.size) < 300:
            raise ValueError(f"image too small: {probe.size}")
        path = "udata/input.jpg"
        probe.convert("RGB").save(path, quality=95)
        INPUT_IMAGE = os.path.abspath(path)
        ATTRIBUTION = {"file": fname, "author": author, "license": lic, "url": url}
        print("fetched:", fname, "| decoded", probe.size, "|", len(r.content), "bytes")
        break
    except Exception as e:
        print("candidate failed:", fname, "->", type(e).__name__, e)

if not INPUT_IMAGE:
    raise SystemExit("No input photo could be downloaded. Re-run this cell, or upload your own (see the cell above).")

INPUT_SHA256 = hashlib.sha256(open(INPUT_IMAGE, "rb").read()).hexdigest()
print("INPUT_IMAGE  :", INPUT_IMAGE)
print("INPUT_SHA256 :", INPUT_SHA256)
print("ATTRIBUTION  :", ATTRIBUTION)
display(Image.open(INPUT_IMAGE))

## 5 · Run InstantMesh inference

Order of attempts (each records its own wall-clock time and peak VRAM):

1. `instant-mesh-large` **with texture map** (`--export_texmap`) → OBJ + MTL + PNG (xatlas UV unwrap)
2. `instant-mesh-base` with texture map — the usual rescue when a 16 GB T4 runs out of memory
3. `instant-mesh-large` **vertex colours only** — fastest, always completes, but *not* textured

The notebook reports exactly which attempt produced the mesh. A "textured" claim is only made if
attempt 1 or 2 succeeded.

In [ ]:
import os, subprocess, time, glob, threading, torch
os.chdir("/content/InstantMesh")

def _vram_sampler(stop, samples):
    """Sample whole-GPU used memory while the child process runs.

    Inference happens in a SUBPROCESS, so torch.cuda.max_memory_allocated() in the
    parent would always read ~0. Sampling nvidia-smi is the honest measurement.
    """
    while not stop.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5)
            for line in out.stdout.strip().splitlines():
                samples.append(float(line))
        except Exception:
            pass
        stop.wait(0.5)


def run_once(config, texmap, timeout_s=5400):
    os.makedirs("outputs", exist_ok=True)
    before = set(glob.glob("outputs/*/meshes/*.obj"))
    cmd = ["python", "run.py", config, INPUT_IMAGE,
           "--output_path", "outputs", "--seed", "42", "--diffusion_steps", "75"]
    if texmap:
        cmd.append("--export_texmap")
    print("$", " ".join(cmd), flush=True)

    torch.cuda.empty_cache()
    try:
        base_vram = float(subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10).stdout.strip().splitlines()[0])
    except Exception:
        base_vram = 0.0

    samples, stop = [], threading.Event()
    sampler = threading.Thread(target=_vram_sampler, args=(stop, samples), daemon=True)
    sampler.start()

    t0 = time.perf_counter()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s,
                              env=dict(os.environ, PYTHONUNBUFFERED="1"))
    finally:
        stop.set(); sampler.join(timeout=5)
    wall = time.perf_counter() - t0

    peak = (max(samples) / 1024.0) if samples else 0.0
    new = sorted(set(glob.glob("outputs/*/meshes/*.obj")) - before)
    tail = (proc.stdout or "")[-1500:] + (proc.stderr or "")[-2500:]
    ok = proc.returncode == 0 and bool(new)
    print(f"  -> rc={proc.returncode} obj={new} wall={wall:.1f}s "
          f"peak_vram={peak:.2f}GiB (baseline {base_vram/1024:.2f}GiB)")
    if not ok:
        print("  --- tail ---"); print(tail)
    return {"config": config, "texmap": texmap, "ok": ok, "wall_s": round(wall, 1),
            "peak_vram_gb": round(peak, 2), "baseline_vram_gb": round(base_vram / 1024, 2),
            "obj": new[-1] if new else None, "returncode": proc.returncode, "tail": tail}


ATTEMPTS = [
    ("configs/instant-mesh-large.yaml", True),
    ("configs/instant-mesh-base.yaml",  True),
    ("configs/instant-mesh-large.yaml", False),
]
ATTEMPT_LOG, WINNER = [], None
for cfg, texmap in ATTEMPTS:
    rec = run_once(cfg, texmap)
    ATTEMPT_LOG.append(rec)
    if rec["ok"]:
        WINNER = rec; break
    torch.cuda.empty_cache()

if not WINNER:
    raise SystemExit("All three attempts failed - see the tails above. Attach this notebook's output to the chat.")

IS_TEXTURED = bool(WINNER["texmap"])
print("\nWINNER:", WINNER["config"], "| textured export:", IS_TEXTURED)
print("generation time:", WINNER["wall_s"], "s | peak VRAM:", WINNER["peak_vram_gb"], "GiB")


## 6 · Convert to GLB (texture preserved) and validate

The official CLI writes **OBJ only**. `trimesh` re-exports it as GLB:
* textured path → GLB embeds the baked PNG + UVs (`TEXCOORD_0`)
* fallback path → GLB keeps per-vertex colours (`COLOR_0`)

Then a structural validation runs *before* you download anything, so a broken file never reaches the chat.

In [ ]:
import os, glob, json, hashlib, trimesh
os.chdir("/content/InstantMesh")

obj_path = WINNER["obj"]
loaded = trimesh.load(obj_path, process=False)

# A textured OBJ loads as a Scene (one geometry per material group), not a Trimesh.
if isinstance(loaded, trimesh.Scene):
    print("loaded a Scene with", len(loaded.geometry), "geometry object(s); concatenating")
    mesh = loaded.dump(concatenate=True)
else:
    mesh = loaded

visual_kind = getattr(getattr(mesh, "visual", None), "kind", "none")
glb_path = obj_path.replace(".obj", ".glb")
mesh.export(glb_path)

mesh_dir = os.path.dirname(obj_path)
siblings = [p for p in glob.glob(os.path.join(mesh_dir, "*")) if not p.endswith((".obj", ".glb"))]
mv_images = sorted(glob.glob("outputs/*/images/*.png"))

validation = {
    "obj_path": obj_path,
    "glb_path": glb_path,
    "glb_bytes": os.path.getsize(glb_path),
    "vertices": int(len(mesh.vertices)),
    "faces": int(len(mesh.faces)),
    "visual_kind": visual_kind,
    "watertight": bool(mesh.is_watertight),
    "bounds": mesh.bounds.tolist(),
    "siblings": [os.path.basename(p) for p in siblings],
    "multiview": [os.path.basename(p) for p in mv_images],
    "glb_sha256": hashlib.sha256(open(glb_path, "rb").read()).hexdigest(),
}
with open(glb_path, "rb") as fh:
    assert fh.read(4) == b"glTF", "exported GLB has a bad magic header"
assert validation["faces"] > 0, "mesh has no faces"

print(json.dumps(validation, indent=2))
if not IS_TEXTURED:
    print("\nNOTE: this mesh is VERTEX-COLOURED, not texture-mapped - say so in the report.")
if validation["visual_kind"] not in ("texture", "color"):
    print("WARNING: visual kind is", validation["visual_kind"], "- the GLB may have no colour data.")


## 7 · Package everything and download

In [ ]:
import os, json, zipfile, glob, base64
os.chdir("/content/InstantMesh")

metrics = {
    "milestone": "real end-to-end image-to-3D",
    "runner": "Google Colab free tier",
    "gpu": info.get("gpu"),
    "vram_total_gb": info.get("vram_total_gb"),
    "torch": info.get("torch"),
    "cuda": info.get("cuda"),
    "input_image": os.path.basename(INPUT_IMAGE),
    "input_sha256": INPUT_SHA256,
    "input_attribution": ATTRIBUTION,
    "pipeline": "official TencentARC/InstantMesh run.py (Zero123++ -> LRM -> FlexiCubes)",
    "diffusion_steps": 75,
    "seed": 42,
    "textured": IS_TEXTURED,
    "winning_config": WINNER["config"],
    "generation_time_s": WINNER["wall_s"],
    "peak_vram_gb": WINNER["peak_vram_gb"],
    "baseline_vram_gb": WINNER["baseline_vram_gb"],
    "vram_measurement": "nvidia-smi memory.used sampled every 0.5s during the child process",
    "attempts": [{k: a[k] for k in ("config", "texmap", "ok", "wall_s", "peak_vram_gb", "baseline_vram_gb", "returncode")}
                 for a in ATTEMPT_LOG],
    "output": validation,
}
open("udata/metrics.json", "w").write(json.dumps(metrics, indent=2))

zip_path = "/content/instantmesh_real_glb.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(INPUT_IMAGE, "input.jpg")
    z.write("udata/metrics.json", "metrics.json")
    for p in [validation["glb_path"], validation["obj_path"]] + \
             [os.path.join(mesh_dir, s) for s in validation["siblings"]] + mv_images:
        if os.path.exists(p):
            z.write(p, os.path.join("mesh", os.path.basename(p)))
print("zip:", zip_path, os.path.getsize(zip_path), "bytes")

print("\n=== COPY THIS BLOCK INTO THE CHAT (it is the evidence) ===")
print(json.dumps(metrics, indent=2))

from google.colab import files
files.download(zip_path)
files.download(validation["glb_path"])

## 8 · What "good" looks like — and known limitations

A successful run gives you, in `instantmesh_real_glb.zip`:

* `mesh/mesh.glb` — the real output, openable in any glTF viewer
* `mesh/mesh.obj` (+ `.mtl` + texture PNG on the textured path)
* `mesh/*.png` — the six generated views; **look at these first**: if the multi-view grid already looks
  wrong, the mesh will be wrong, and the fix is a different `--seed`.
* `input.jpg` + `metrics.json`

**Expected for a car (be realistic about a 2024 single-image model):**
* body proportions and stance usually good; wheels often soft or slightly elliptical
* the **underside is invented** (never visible in the photo)
* mirrors, wipers, grille slats and door seams get blurred or lost
* glossy paint is baked as flat colour — no reflections, no PBR maps
* a texture map is not a guarantee of crisp text; the M5 badge will not be legible

**If the result is poor:** re-run cell 5 with a different `--seed` (e.g. 7 or 1234), or try a cleaner
side-on photo. Cars are hard-mode for this class of model.